## 모기업 상위 100개 리스트

In [2]:
from pathlib import Path

import pandas as pd


# 프로젝트 경로 설정
PROJECT_ROOT = Path.home() / "projects" / "mle-01-p2-team3"
DATA_DIR = PROJECT_ROOT / "data" / "clean"

print("현재 노트북 위치:", Path.cwd())
print("데이터 폴더 존재:", DATA_DIR.exists())


# 1. 기업 기본정보 불러오기
company_df = pd.read_json(
    DATA_DIR / "기업개요_최종.jsonl",
    lines=True,
    dtype={"crno": str},
)


# 2. 관계 통합 파일에서 모기업만 고유하게 추출
relation_df = pd.read_csv(
    DATA_DIR / "모기업_계열사_종속기업_통합.csv",
    dtype={"top_crno": str},
)

parent_df = (
    relation_df[["top_crno", "top_corpNm"]]
    .drop_duplicates(subset="top_crno")
    .rename(
        columns={
            "top_crno": "crno",
            "top_corpNm": "company_name",
        }
    )
)


# 3. 모기업 목록과 기업 기본정보 연결
target_df = parent_df.merge(
    company_df[
        [
            "crno",
            "enpEmpeCnt",
            "enpBsadr",
            "enpHmpgUrl",
            "상장여부",
            "top_region",
        ]
    ],
    on="crno",
    how="left",
)


# 4. 직원 수를 숫자로 변환
target_df["employee_count"] = pd.to_numeric(
    target_df["enpEmpeCnt"]
    .astype(str)
    .str.replace(",", "", regex=False),
    errors="coerce",
).fillna(0)


# 5. 직원 수 기준 상위 100개 모기업 선정
top100_df = (
    target_df
    .sort_values("employee_count", ascending=False)
    .head(100)
    .reset_index(drop=True)
)


# 6. 뉴스 수집 대상 기업 목록 저장
output_path = DATA_DIR / "news_target_companies_top100.csv"

top100_df.to_csv(
    output_path,
    index=False,
    encoding="utf-8-sig",
)


# 7. 결과 확인
print(f"\n저장 완료: {output_path}")
print(f"선정 기업 수: {len(top100_df)}개\n")

display(
    top100_df[
        [
            "crno",
            "company_name",
            "employee_count",
            "top_region",
            "상장여부",
            "enpHmpgUrl",
        ]
    ].head(20)
)

현재 노트북 위치: /Users/gh/projects/mle-01-p2-team3/notebooks
데이터 폴더 존재: True

저장 완료: /Users/gh/projects/mle-01-p2-team3/data/clean/news_target_companies_top100.csv
선정 기업 수: 100개



,crno,company_name,employee_count,top_region,상장여부,enpHmpgUrl
0,1101110085450,현대자동차(주),72598.0,서울,1.0,www.hyundai.com
1,1101110037998,기아(주),36566.0,서울,1.0,www.kia.co.kr
2,1101110393134,엘지디스플레이(주),24430.0,서울,1.0,www.lgdisplay.com
3,1101110108484,(주)대한항공,18318.0,서울,1.0,www.koreanair.com
4,1101110000086,롯데쇼핑(주),17712.0,서울,1.0,www.lotteshoppingir.com/
5,1101110023393,(주)우리은행,14203.0,서울,0.0,www.wooribank.com
6,1101110394174,삼성SDI(주),12826.0,경기,1.0,www.samsungsdi.co.kr
7,1101110012809,(주)신한은행,12781.0,서울,0.0,www.shinhan.com
8,1101110215536,현대모비스(주),12535.0,서울,1.0,www.mobis.com
9,1101110192180,엘지이노텍(주),12211.0,서울,1.0,www.lginnotek.co.kr


## 파일 저장

In [3]:
from pathlib import Path
import json

PROJECT_ROOT = Path.home() / "projects" / "mle-01-p2-team3"
DATA_DIR = PROJECT_ROOT / "data" / "clean"

input_path = DATA_DIR / "news_articles.jsonl"
output_path = DATA_DIR / "news_articles_for_embedding.jsonl"

news_rows = []

with input_path.open("r", encoding="utf-8") as file:
    for line in file:
        if not line.strip():
            continue

        row = json.loads(line)

        news_rows.append(
            {
                "id": row["record_id"],
                "company_id": row["company_crno"],
                "date": row["published_at"],
                "title": row["title"],
                "body": "",
                "summary": row["description"],
                "embedding": None,
            }
        )

with output_path.open("w", encoding="utf-8") as file:
    for row in news_rows:
        file.write(json.dumps(row, ensure_ascii=False) + "\n")

print(f"저장 완료: {output_path}")
print(f"뉴스 수: {len(news_rows)}건")
print(json.dumps(news_rows[0], ensure_ascii=False, indent=2))

저장 완료: /Users/gh/projects/mle-01-p2-team3/data/clean/news_articles_for_embedding.jsonl
뉴스 수: 380건
{
  "id": "fcafd48df4d3121f0b681eee26dc79e7628227fea6126e0602b9f1b7263226d2",
  "company_id": "1101110085450",
  "date": "Thu, 17 Sep 2026 13:36:00 +0900",
  "title": "현대자동차직업전문학교, '과정평가형 자동차정비산업기사' 과정 10월 개...",
  "body": "",
  "summary": "대전 현대자동차직업전문학교(이사장 유성식)가 미래 자동차정비산업기사 인재 양성을 위해 대전RSC와 협력해 자동차 기업 및 지역 유관기관과의 산학협력 체계를 확대하고 있다. 대전 현대직업전문학교는 대전RSC와 함께...",
  "embedding": null
}


In [4]:
from pathlib import Path
import json

path = Path.home() / "projects" / "mle-01-p2-team3" / "data" / "clean" / "news_articles_for_embedding.jsonl"

rows = [
    json.loads(line)
    for line in path.read_text(encoding="utf-8").splitlines()
    if line.strip()
]

print("뉴스 수:", len(rows))
print("필드:", list(rows[0].keys()))
print("제목 없는 뉴스:", sum(not row["title"] for row in rows))
print("요약 없는 뉴스:", sum(not row["summary"] for row in rows))

뉴스 수: 380
필드: ['id', 'company_id', 'date', 'title', 'body', 'summary', 'embedding']
제목 없는 뉴스: 0
요약 없는 뉴스: 0


In [5]:
from pathlib import Path
import json
import re

import pandas as pd


PROJECT_ROOT = Path.home() / "projects" / "mle-01-p2-team3"
DATA_DIR = PROJECT_ROOT / "data" / "clean"

news_path = DATA_DIR / "news_articles_for_embedding.jsonl"
company_path = DATA_DIR / "기업개요_최종.csv"


def clean_company_name(name: str) -> str:
    """(주), (유), ㈜, 주식회사 등의 법인 표기를 제거한다."""
    name = str(name).strip()

    remove_tokens = [
        "(주)",
        "(유)",
        "(사)",
        "(재)",
        "㈜",
        "㈔",
        "㈜",
        "주식회사",
        "유한회사",
    ]

    for token in remove_tokens:
        name = name.replace(token, "")

    return re.sub(r"\s+", " ", name).strip()


# 기업개요 CSV: crno -> 정제된 기업명 사전 생성
company_df = pd.read_csv(
    company_path,
    dtype={"crno": str},
    encoding="utf-8-sig",
)

company_name_map = {
    row["crno"]: clean_company_name(row["corpNm"])
    for _, row in company_df.iterrows()
}


# 기존 뉴스 JSONL 읽기
news_rows = []

with news_path.open("r", encoding="utf-8") as file:
    for line in file:
        if line.strip():
            news_rows.append(json.loads(line))


# company_name 추가
updated_rows = []
missing_company_ids = set()

for row in news_rows:
    company_id = row["company_id"]
    company_name = company_name_map.get(company_id)

    if company_name is None:
        missing_company_ids.add(company_id)
        company_name = ""

    updated_rows.append(
        {
            "id": row["id"],
            "company_id": company_id,
            "company_name": company_name,
            "date": row["date"],
            "title": row["title"],
            "body": row["body"],
            "summary": row["summary"],
            "embedding": row["embedding"],
        }
    )


# 기존 파일을 새 구조로 갱신
with news_path.open("w", encoding="utf-8") as file:
    for row in updated_rows:
        file.write(json.dumps(row, ensure_ascii=False) + "\n")


print(f"저장 완료: {news_path}")
print(f"뉴스 수: {len(updated_rows)}건")
print(f"기업명을 찾지 못한 company_id 수: {len(missing_company_ids)}")

if missing_company_ids:
    print("누락 company_id 예시:", list(missing_company_ids)[:10])

print("\n샘플:")
print(json.dumps(updated_rows[0], ensure_ascii=False, indent=2))

저장 완료: /Users/gh/projects/mle-01-p2-team3/data/clean/news_articles_for_embedding.jsonl
뉴스 수: 380건
기업명을 찾지 못한 company_id 수: 0

샘플:
{
  "id": "fcafd48df4d3121f0b681eee26dc79e7628227fea6126e0602b9f1b7263226d2",
  "company_id": "1101110085450",
  "company_name": "현대자동차",
  "date": "Thu, 17 Sep 2026 13:36:00 +0900",
  "title": "현대자동차직업전문학교, '과정평가형 자동차정비산업기사' 과정 10월 개...",
  "body": "",
  "summary": "대전 현대자동차직업전문학교(이사장 유성식)가 미래 자동차정비산업기사 인재 양성을 위해 대전RSC와 협력해 자동차 기업 및 지역 유관기관과의 산학협력 체계를 확대하고 있다. 대전 현대직업전문학교는 대전RSC와 함께...",
  "embedding": null
}
